# Step 6.1: Environment Setup and Hardware Configuration

## 1. Motivation for Transfer Learning with EfficientNetV2S

The previous step established Xception as a solid transfer learning baseline, meaningfully outperforming the from-scratch CNN experiments on the WikiArt dataset. This notebook builds on that result by replacing the backbone with EfficientNetV2S (a more recent architecture) to evaluate whether a stronger pretrained feature extractor translates into a measurable gain in F1-score on the art style classification task.

EfficientNetV2S was selected for several reasons. Its compound scaling approach jointly optimises network depth, width, and resolution, yielding strong accuracy-to-parameter ratios compared to Xception and heavier alternatives like ResNet-152 or VGG. The V2 architecture replaces early depthwise separable convolutions with Fused-MBConv blocks, improving training speed and gradient flow on smaller datasets. Crucially, the 384×384 native resolution (larger than Xception's 299×299) should preserve finer brushstroke and texture detail that is central to distinguishing art styles. The expectation is that both the architectural improvements and the higher input resolution will compound on the Xception baseline and push the model further.

In [1]:
import os
from pathlib import Path
import json
import math
from keras import Model, layers
from keras.applications import EfficientNetV2S
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
from keras.utils import image_dataset_from_directory

## 2. Hardware Optimisation

Implements `set_memory_growth` to prevent TensorFlow from allocating the entirety of the VRAM at startup, avoiding hard crashes during execution.

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")

# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
# tf.config.optimizer.set_jit(True)
# print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']


# Step 6.2: Transfer Learning Architecture - EfficientNetV2S

## 1. Model Design Philosophy

Defines `TransferEfficientNetV2S`, a Keras `Model` subclass wrapping the pretrained EfficientNetV2S backbone with a lightweight classification head. EfficientNetV2S applies internal rescaling and normalisation natively, so no external `Rescaling` layer is needed. Inputs are passed directly from the data pipeline. Augmentation is handled entirely externally via the `tf.data` pipeline using Albumentations, keeping the model graph clean and serialisation straightforward.

## 2. Two-Phase Trainability Design

The constructor freezes `self.base` entirely at initialisation, enabling a clean Phase 1 where only the newly added head (Global Average Pooling, Dropout, and a softmax Dense layer) is updated against the ImageNet-pretrained representations. The `unfreeze_base` method implements Phase 2: it sets the base to trainable and then iterates over its layers, re-freezing the first `n_freeze` layers (which capture low-level, domain-agnostic features such as edges and textures) and unconditionally re-freezing all Batch Normalisation layers throughout the network to prevent running-statistic drift from destabilising training on the art domain. A diagnostic print reports the frozen/unfrozen split so the configuration can be verified at runtime.

## 3. Configuration Serialisation

The `get_config` method extends the parent class configuration with the model's custom constructor arguments (`num_classes`, `dropout_rate`), ensuring the model can be correctly reconstructed from a saved checkpoint without manual argument tracking.


In [3]:
class TransferEfficientNetV2S(Model):
    """
    Pre-trained EfficientNetV2S.
    Note: EfficientNetV2 models include internal rescaling/normalisation.
    Augmentation is handled externally via tf.data.Dataset (Albumentations).
    """

    def __init__(self, num_classes, dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="transfer_effnetv2s")
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        self.base = EfficientNetV2S(
            include_top=False, 
            weights='imagenet' # Ensure weights are loaded
        )

        # Freeze the base model if you only want to train the head initially
        self.base.trainable = False 

        self.gap_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(dropout_rate)
        self.dense_layer = layers.Dense(self.num_classes, activation="softmax")

    def unfreeze_base(self, n_freeze=350):
        """
        Phase 2: unfreeze the top layers of the base for fine-tuning.
        n_freeze: number of early layers to keep frozen (they learn generic features
                  that transfer well and don't need retraining).
        """
        self.base.trainable = True
        for i, layer in enumerate(self.base.layers):
            # RULE A: Freeze the first N layers (low-level features)
            if i < n_freeze:
                layer.trainable = False
            
            # RULE B: Freeze ALL Batch Normalization layers (for Stability)
            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        frozen = sum(1 for l in self.base.layers if not l.trainable)
        total  = len(self.base.layers)
        print(f"{self.name}: {frozen}/{total} base layers frozen, {total - frozen} unfrozen")

    def get_config(self):
        # Obtain the base config from the parent class
        config = super().get_config()
        # Add custom parameters to the config
        config.update({
            "num_classes": self.num_classes,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # Pass inputs directly to EfficientNet (it will rescale them internally)
        x = self.base(inputs, training=training)

        x = self.gap_layer(x)
        x = self.dropout_layer(x, training=training)
        return self.dense_layer(x)

# Step 6.3: Hyperparameters and Data Pipeline

## 1. Global Configuration

Establishes the critical training parameters for both phases. The image resolution is set to 384×384, the native resolution for EfficientNetV2S, this yields approximately 2.9× more pixels per image compared to 224×224, providing a significant accuracy gain at the cost of higher memory consumption; the batch size of 16 should be reduced to 8 if out-of-memory errors are encountered on an 8 GB GPU. Two distinct learning rates are defined: `PHASE1_LR` at 1e-3 for the warm head training and `PHASE2_LR` at 1e-5 (approximately 100× lower) to fine-tune the unfrozen base layers without overwriting the pretrained representations. Directories for checkpoints and training metrics are created idempotently.

## 2. Dataset Instantiation and Mixup

Loads the train, validation, and test splits from the partitioned `wikiart_split` directory using `image_dataset_from_directory`, applying `crop_to_aspect_ratio=True` to avoid distortion when resizing to the square target resolution. Categorical label encoding is used throughout to match the softmax output format.

A `mixup` function with `alpha=0.4` is defined to blend pairs of images and their labels proportionally, acting as a data-space regulariser that discourages overconfident predictions. It is applied to the training dataset via `map()` with `AUTOTUNE`-parallelised execution, followed by prefetching to overlap preprocessing with GPU computation. The validation and test datasets are left unaugmented to ensure evaluation reflects true generalisation.

In [4]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# 384×384: native resolution for EfficientNetV2S (significant accuracy gain over 224)
# Note: ~2.9× more pixels per image — reduce batch_size if you hit OOM on GPU
IMAGE_SIZE     = (384, 384)
BATCH_SIZE     = 16       # adjust based on your GPU's VRAM (e.g., 8 or 16 for 8GB, 32+ for 16GB)
PHASE1_EPOCHS  = 25       # frozen-base head training
PHASE2_EPOCHS  = 40       # fine-tuning (EarlyStopping will cut this short)
PHASE1_LR      = 1e-3     # higher LR — only head is updating
PHASE2_LR      = 1e-5     # ~100× lower LR — prevent destroying pretrained weights
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# 1. Load raw images (batched) from directories
train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
def mixup(images, labels, alpha=0.4):
    images = tf.cast(images, tf.float32)
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

# Applies mixup augmentation to the training dataset.
# We use map() to apply the mixup function to each batch of images and labels.
# The num_parallel_calls=AUTOTUNE argument allows TensorFlow to determine the optimal number of parallel calls for performance.
# Finally, we call prefetch(AUTOTUNE) to allow the dataset to fetch batches in the background while the model is training, improving performance.
train_ds_mixed = train_ds.map(mixup, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


# Step 6.4: Class Weights, Model Instantiation, and Metrics

## 1. Class Weights

Loads the pre-computed class weights from `class_weights.json`. These weights compensate for the significant class imbalance in the WikiArt dataset by scaling the loss contribution of under-represented art styles upward, preventing the model from achieving low training loss simply by predicting the majority classes.

In [ ]:
# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}

## 2. Model Instantiation

Calls `clear_session()` before instantiating the model to flush any residual graph state and variable allocations from previous runs in the same kernel session, freeing GPU memory cleanly. The model is instantiated with the 23-class output configuration at the default dropout rate of 0.5.

In [6]:
clear_session() # Clear previous models from memory before instantiating new ones.

model = TransferEfficientNetV2S(num_classes=N_CLASSES)

## 3. Metrics and Loss

Defines a `make_metrics` factory function that returns a fresh set of stateful metric instances each time it is called. This is necessary because Keras metrics accumulate state across batches, and reusing the same instances across compilation calls would contaminate Phase 2 evaluation with Phase 1 statistics. The metric set comprises Categorical Accuracy, AUC (multi-label formulation), and Macro F1-score via `tfa.metrics.F1Score`, providing a balanced assessment across all 23 art style classes.

In [ ]:
def make_metrics(num_classes):
    """Return a fresh set of metric instances (metrics are stateful — each model needs its own)."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]

# Step 6.5: Learning Rate Schedule

## 1. Cosine Annealing with Linear Warmup

Defines a `make_cosine_warmup_scheduler` factory that returns a `LearningRateScheduler`-compatible function combining two regimes. During the warmup phase (the first `warmup_epochs` epochs), the learning rate rises linearly from 0 to `base_lr`. This prevents the randomly initialised classification head from producing large gradients that could damage the pretrained feature representations before the head has learned to produce reasonable outputs. After warmup, the learning rate follows a cosine decay curve from `base_lr` down to approximately 0, which tends to find better minima than step-decay or exponential schedules by smoothly annealing the optimisation landscape rather than abruptly reducing the step size.

In [ ]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Cosine annealing with linear warmup.

    Warmup: LR ramps linearly from 0 to base_lr over the first warmup_epochs.
    This prevents the randomly initialised head from producing large gradients
    that destabilise the pretrained base at the start of training.

    Cosine decay: LR then follows a cosine curve from base_lr down to ~0.
    Finds better minima than step-decay or exponential decay in practice.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler

# Step 6.6: Phase 1 - Train Head with Frozen Base

## 1. Head Training Strategy

In Phase 1 the pretrained EfficientNetV2S base is completely frozen and only the GAP + Dropout + Dense head is updated. This is the standard transfer learning warm-up: the head is randomly initialised and would otherwise produce destructively large gradients if the base were simultaneously trainable. By isolating the head update for the first `PHASE1_EPOCHS` epochs, the model learns to map EfficientNet features to the 23 WikiArt classes without disturbing the ImageNet representations.

## 2. Compilation and Callbacks

Compiled with `AdamW` at `PHASE1_LR` and a conservative weight decay of 1e-6, `CategoricalCrossentropy` with `label_smoothing=0.1` to discourage overconfident softmax outputs, and fresh metrics from `make_metrics`. The callback stack saves the best checkpoint by validation loss, logs per-epoch metrics to CSV, applies the cosine warmup schedule (3 warmup epochs), and stops early with a patience of 5 epochs to avoid wasting compute if the head converges prematurely.

In [9]:
print(f"\n{'='*60}")
print(f"Phase 1 training: {model.name}")
print(f"{'='*60}")


model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE1_LR, weight_decay=1e-6),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase1_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase1_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE1_LR, PHASE1_EPOCHS, warmup_epochs=3)
    ),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase1_fit_data = history

print("\nPhase 1 complete.")



Phase 1 training: transfer_effnetv2s
Epoch 1/25
583/583 [==============================] - ETA: 0s - loss: 2.8117 - accuracy: 0.2583 - auc: 0.6624 - f1_score: 0.2053
Epoch 1: val_loss improved from inf to 2.17982, saving model to Checkpoints\ckpt_phase1_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


583/583 [==============================] - 253s 405ms/step - loss: 2.8117 - accuracy: 0.2583 - auc: 0.6624 - f1_score: 0.2053 - val_loss: 2.1798 - val_accuracy: 0.4905 - val_auc: 0.9111 - val_f1_score: 0.4601 - lr: 3.3333e-04
Epoch 2/25
583/583 [==============================] - ETA: 0s - loss: 2.3892 - accuracy: 0.4542 - auc: 0.7376 - f1_score: 0.3712
Epoch 2: val_loss improved from 2.17982 to 1.83027, saving model to Checkpoints\ckpt_phase1_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


583/583 [==============================] - 218s 373ms/step - loss: 2.0660 - accuracy: 0.6122 - auc: 0.7797 - f1_score: 0.5161 - val_loss: 1.5092 - val_accuracy: 0.7214 - val_auc: 0.9673 - val_f1_score: 0.7022 - lr: 1.7257e-04
Epoch 21/25
583/583 [==============================] - ETA: 0s - loss: 2.0391 - accuracy: 0.6225 - auc: 0.7782 - f1_score: 0.5243
Epoch 21: val_loss improved from 1.50925 to 1.50365, saving model to Checkpoints\ckpt_phase1_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


583/583 [==============================] - 219s 375ms/step - loss: 2.0391 - accuracy: 0.6225 - auc: 0.7782 - f1_score: 0.5243 - val_loss: 1.5036 - val_accuracy: 0.7214 - val_auc: 0.9676 - val_f1_score: 0.7007 - lr: 1.2213e-04
Epoch 22/25
583/583 [==============================] - ETA: 0s - loss: 2.0532 - accuracy: 0.6186 - auc: 0.7763 - f1_score: 0.5219
Epoch 22: val_loss did not improve from 1.50365
583/583 [==============================] - 129s 220ms/step - loss: 2.0532 - accuracy: 0.6186 - auc: 0.7763 - f1_score: 0.5219 - val_loss: 1.5058 - val_accuracy: 0.7184 - val_auc: 0.9678 - val_f1_score: 0.6977 - lr: 7.9373e-05
Epoch 23/25
583/583 [==============================] - ETA: 0s - loss: 2.0462 - accuracy: 0.6206 - auc: 0.7818 - f1_score: 0.5211
Epoch 23: val_loss did not improve from 1.50365
583/583 [==============================] - 129s 220ms/step - loss: 2.0462 - accuracy: 0.6206 - auc: 0.7818 - f1_score: 0.5211 - val_loss: 1.5042 - val_accuracy: 0.7204 - val_auc: 0.9680 - val_

# Step 6.7: Phase 1 Evaluation

## 1. Baseline Performance

Evaluates the Phase 1 checkpoint  against the held-out test set to establish a clean baseline before the base network is unfrozen. This score reflects how well the frozen EfficientNetV2S features alone, combined with the newly trained head, generalise to unseen WikiArt images, and serves as the lower bound against which Phase 2 fine-tuning will be assessed.

In [10]:
phase1_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase1_eval_data

{'loss': 1.5073416233062744,
 'accuracy': 0.7255192995071411,
 'auc': 0.9678757786750793,
 'f1_score': 0.70982825756073}

# Step 6.8: Phase 2 - Fine-tune Unfrozen Base Layers

## 1. Fine-tuning Strategy

Phase 2 unfreezes the top portion of the pretrained base and retrains the entire unfrozen network at a much lower learning rate. The underlying intuition is that early layers of EfficientNetV2S capture generic low-level features (edges, colour gradients, textures) that transfer well across visual domains and should remain frozen, whilst later layers encode higher-level semantic patterns that are more domain-specific and can be productively adapted to the stylistic vocabulary of the WikiArt dataset.

## 2. Recompilation and Callbacks

The base is unfrozen via `model.unfreeze_base()` using the default `n_freeze=350`, preserving the early convolutional layers. The model is recompiled at `PHASE2_LR` (1e-5, approximately 100× lower than Phase 1) with a weight decay of 1e-7 to apply very gentle L2 regularisation whilst avoiding overwriting the pretrained representations. EarlyStopping patience is increased to 10 epochs; Improvements in fine-tuning are smaller and more gradual than in head training, so more patience is needed to distinguish genuine plateaus from temporary fluctuations.

In [11]:
print(f"\n{'='*60}")
print(f"Phase 2 fine-tuning: {model.name}")
print(f"{'='*60}")

# Unfreeze top layers — defaults are set inside each model class
model.unfreeze_base()

# Recompile at ~100× lower LR to avoid overwriting pretrained representations
model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE2_LR, weight_decay=1e-7),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase2_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase2_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE2_LR, PHASE2_EPOCHS, warmup_epochs=2)
    ),
    # More patience in Phase 2 — improvements are smaller and slower
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase2_fit_data = history

print("\nPhase 2 complete.")



Phase 2 fine-tuning: transfer_effnetv2s
transfer_effnetv2s: 382/513 base layers frozen, 131 unfrozen
Epoch 1/40
583/583 [==============================] - ETA: 0s - loss: 1.9993 - accuracy: 0.6410 - auc: 0.7807 - f1_score: 0.5421
Epoch 1: val_loss improved from inf to 1.45458, saving model to Checkpoints\ckpt_phase2_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


583/583 [==============================] - 286s 464ms/step - loss: 1.9993 - accuracy: 0.6410 - auc: 0.7807 - f1_score: 0.5421 - val_loss: 1.4546 - val_accuracy: 0.7405 - val_auc: 0.9723 - val_f1_score: 0.7213 - lr: 5.0000e-06
Epoch 2/40
583/583 [==============================] - ETA: 0s - loss: 1.9598 - accuracy: 0.6620 - auc: 0.7848 - f1_score: 0.5599
Epoch 2: val_loss improved from 1.45458 to 1.40863, saving model to Checkpoints\ckpt_phase2_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


583/583 [==============================] - 278s 477ms/step - loss: 1.5727 - accuracy: 0.8562 - auc: 0.8132 - f1_score: 0.7275 - val_loss: 1.1221 - val_accuracy: 0.8424 - val_auc: 0.9892 - val_f1_score: 0.8264 - lr: 1.5300e-07
Epoch 39/40
583/583 [==============================] - ETA: 0s - loss: 1.5771 - accuracy: 0.8580 - auc: 0.8154 - f1_score: 0.7284
Epoch 39: val_loss improved from 1.12212 to 1.12203, saving model to Checkpoints\ckpt_phase2_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


583/583 [==============================] - 268s 459ms/step - loss: 1.5771 - accuracy: 0.8580 - auc: 0.8154 - f1_score: 0.7284 - val_loss: 1.1220 - val_accuracy: 0.8424 - val_auc: 0.9892 - val_f1_score: 0.8264 - lr: 6.8193e-08
Epoch 40/40
583/583 [==============================] - ETA: 0s - loss: 1.5874 - accuracy: 0.8575 - auc: 0.8170 - f1_score: 0.7234
Epoch 40: val_loss did not improve from 1.12203
583/583 [==============================] - 180s 309ms/step - loss: 1.5874 - accuracy: 0.8575 - auc: 0.8170 - f1_score: 0.7234 - val_loss: 1.1220 - val_accuracy: 0.8424 - val_auc: 0.9892 - val_f1_score: 0.8264 - lr: 1.7078e-08

Phase 2 complete.


# Step 6.9: Phase 2 Evaluation and Final Results

## 1. Final Performance Assessment

Evaluates the best Phase 2 checkpoint against the held-out test set to produce the definitive performance figures for this model. Comparing these metrics against the Phase 1 baseline quantifies the gain attributable to fine-tuning the unfrozen base layers on the art domain, and the results are stored in `phase2_eval_data` for downstream reporting and cross-model comparison.

In [12]:
phase2_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase2_eval_data

{'loss': 1.118994116783142,
 'accuracy': 0.8456973433494568,
 'auc': 0.9875801801681519,
 'f1_score': 0.837436318397522}